In [ ]:
path_extracted_data = "Data\\Extracted Arguments No Keywords (all text)\\"
prefix = 'OCDE_2024_'
model_name="qwen2.5-3b"

### Git push to save folder structure

**Push**


```
!git add .
!git commit -m "Commit"
!git push origin main  # or 'master' or your branch name
```



# Generate unified .json with all extracted arguments 

- Clean duplicated arguments 
- Display summary statistics per SDG (totals/means, duplicates removed totals/means)
- Display summary statistics (same metrics over the whole set)

In [ ]:
import os, re, json
from collections import defaultdict, OrderedDict
import pandas as pd
from typing import List, Optional, Tuple

def process_sdg_runs(
    input_dir: str,
    output_dir: str,
    prefix: str,
    model_name: Optional[str] = None,
    ods_range: Tuple[int, int] = (0, 17),
    unified_basename: Optional[str] = None,
):

    os.makedirs(output_dir, exist_ok=True)

    # 1. match files of interest
    model_part = re.escape(model_name) if model_name else r".+?"
    pattern = re.compile(rf"^{re.escape(prefix)}_ArgsSGD(\d+)_({model_part})\.json$")

    files = []
    for fn in os.listdir(input_dir):
        m = pattern.match(fn)
        if not m:
            continue
        ods_num = int(m.group(1))
        if ods_num < ods_range[0] or ods_num > ods_range[1]:
            continue
        files.append((ods_num, fn))

    if not files:
        raise FileNotFoundError(f"No files found in '{input_dir}' for prefix='{prefix}'"
                                + (f" and model='{model_name}'" if model_name else ""))

    # 2. Process per ODS (each .json file)
    files.sort(key=lambda x: (x[0], x[1]))

    unified_records = []
    per_sdg_stats = []

    # Overall stats
    overall_pages = 0
    overall_total_kept = 0
    overall_total_removed = 0
    overall_total_original = 0

    for ods_num, fn in files:
        path = os.path.join(input_dir, fn)
        with open(path, "r", encoding="utf-8") as f:
            pages = json.load(f)

        seen_args = OrderedDict()

        # Per-page stats
        page_counts_kept = []
        page_counts_removed = []
        page_counts_original = []

        # 2.1 Process pages
        for item in pages:
            page = item.get("page")
            section = item.get("section")
            args: List[str] = item.get("arguments", []) or []

            def unique_in_order(seq):
                seen = set()
                out = []
                for s in seq:
                    if s not in seen:
                        out.append(s)
                        seen.add(s)
                return out

            args_original_unique = unique_in_order(args)
            page_original = len(args_original_unique)

            #3. Deduplicate across the entire ODS (per file remove arguments extracted redundantly)
            kept_args = []
            removed_here = 0
            for a in args_original_unique:
                if a not in seen_args:
                    kept_args.append(a)
                    seen_args[a] = True
                else:
                    removed_here += 1

            #4. Save unified record (even if empty arguments list, to retain page info)
            unified_records.append({
                "ods": ods_num,
                "page": page,
                "section": section,
                "arguments": kept_args,
                "source_file": fn,
            })

            # Per-page stats
            page_counts_original.append(page_original)
            page_counts_kept.append(len(kept_args))
            page_counts_removed.append(removed_here)

        # Per-SDG summary row
        pages_count = len(pages)
        total_kept = sum(page_counts_kept)
        total_removed = sum(page_counts_removed)
        total_original = sum(page_counts_original)

        mean_kept = (total_kept / pages_count) if pages_count else 0.0
        mean_removed = (total_removed / pages_count) if pages_count else 0.0
        mean_original = (total_original / pages_count) if pages_count else 0.0
        
        #5.1 Final summary per SGD stats
        per_sdg_stats.append({
            "prefix": prefix,
            "model": model_name if model_name else "mixed",
            "ods": ods_num,
            "pages": pages_count,
            "total_args_kept": total_kept,
            "mean_args_kept_per_page": round(mean_kept, 3),
            "total_dupes_removed": total_removed,
            "mean_dupes_removed_per_page": round(mean_removed, 3),
            "total_args_original": total_original,
            "mean_args_original_per_page": round(mean_original, 3),
        })

        # Overall stats
        overall_pages += pages_count
        overall_total_kept += total_kept
        overall_total_removed += total_removed
        overall_total_original += total_original
    per_sdg_df = pd.DataFrame(per_sdg_stats).sort_values(["ods"]).reset_index(drop=True)

    #5.2 Final summary stats
    overall_summary = pd.DataFrame([{
        "prefix": prefix,
        "model": model_name if model_name else "mixed",
        "ods_count": len(per_sdg_df),
        "pages_total": overall_pages,
        "total_args_kept": overall_total_kept,
        "mean_args_kept_per_page": round((overall_total_kept / overall_pages) if overall_pages else 0.0, 3),
        "total_dupes_removed": overall_total_removed,
        "mean_dupes_removed_per_page": round((overall_total_removed / overall_pages) if overall_pages else 0.0, 3),
        "total_args_original": overall_total_original,
        "mean_args_original_per_page": round((overall_total_original / overall_pages) if overall_pages else 0.0, 3),
    }])

    #6. Save outputs
    if unified_basename is None:
        unified_basename = f"{prefix}_Unified_{model_name}"

    unified_json_path = os.path.join(output_dir, f"{unified_basename}.json")
    per_sdg_csv_path = os.path.join(output_dir, f"{unified_basename}_perSDG_summary.csv")
    overall_csv_path = os.path.join(output_dir, f"{unified_basename}_overall_summary.csv")

    with open(unified_json_path, "w", encoding="utf-8") as f:
        json.dump(unified_records, f, indent=2, ensure_ascii=False)

    per_sdg_df.to_csv(per_sdg_csv_path, index=False, encoding="utf-8")
    overall_summary.to_csv(overall_csv_path, index=False, encoding="utf-8")


    print(f"\nProcessed {len(per_sdg_df)} SDGs for prefix='{prefix}'"
          + (f", model='{model_name}'" if model_name else ", mixed models"))
    print(per_sdg_df[["ods", "pages", "total_args_kept", "mean_args_kept_per_page",
                      "total_dupes_removed", "mean_dupes_removed_per_page"]].to_string(index=False))
    print("\nOverall summary:")
    print(overall_summary.to_string(index=False))

    return {
        "unified_json": unified_json_path,
        "per_sdg_summary_csv": per_sdg_csv_path,
        "overall_summary_csv": overall_csv_path,
        "per_sdg_summary_df": per_sdg_df,      
        "overall_summary_df": overall_summary,
    }


## GLOBAL SDG 2023 report

### Define relevant sections to extract text from the Full report


| Section Title                                      | Pages  |
| -------------------------------------------------- | ------ |
| 1\_Executive Summary                               | 2      |
| 2\_How to Achieve the SDGs: The SDSN Framework     | 20     |
| 3\_The SDG Index and Dashboards                    | 6      |
| 4\_Government Efforts and Commitments for the SDGs | 12     |
| 5\_Lessons Learned and Next Steps                  | 13     |
| **Total**                                          | **53** |


In [9]:
file_name = "GLOBAL_sustainable-development-report-2023.pdf"
prefix = "GLOBAL_SGD2023_"

phrases_to_remove = [
    "\nSustainable Development Report 2023",
    "\n   Implementing the SDG Stimulus"
]

In [10]:
relevant_sections = {
    "1_Executive Summary": {"start_page": 7, "end_page": 8},
    "2_How to Achieve the SDGs: The SDSN Framework": {"start_page": 13, "end_page": 32},
    "3_The SDG Index and Dashboards": {"start_page": 34, "end_page": 46},
    "4_Government Efforts and Commitments for the SDGs": {"start_page": 58, "end_page": 82},
    "5_Lessons Learned and Next Steps": {"start_page": 88, "end_page": 100}
}

# Ensure the directory exists
os.makedirs(path_raw_data, exist_ok=True)

# Save dictionary to JSON file
json_path = os.path.join(path_raw_data, prefix + "sections_metadata.json")
with open(json_path, "w") as f:
    json.dump(relevant_sections, f, indent=4)

### Extract relevant text by section in json with page and text

In [11]:
sections_metadata_file = os.path.join(path_raw_data, prefix + "sections_metadata.json")
pdf_path = os.path.join(path_raw_data, file_name)

# Output directory
output_dir = os.path.join(path_data, "Processed Files (sections)")
os.makedirs(output_dir, exist_ok=True)

# Generate .json files by section
extract_relevant_text_save_by_section(pdf_path, sections_metadata_file, output_dir,
                                      prefix=prefix,
                                      phrases_to_remove=phrases_to_remove)

Extraction complete. JSON files saved in: /content/TFM/Data/Processed Files (sections)


In [12]:
### Manual corrections (exclude pages with only tables or infographics to avoid wrong interpretations)

# Pages to exclude per section
pages_to_exclude = {
    "3_The SDG Index and Dashboards": [35, 36, 37, 39, 40, 41, 46],
    "4_Government Efforts and Commitments for the SDGs": [60, 61, 63, 64, 67, 68, 70, 73, 75, 77, 78, 79, 80]
}

def section_to_filename(section_title):
    return section_title.replace(" ", "_").replace(":", "").replace("/", "-") + ".json"

# Process each targeted section
for section, exclude_pages in pages_to_exclude.items():
    file_name = prefix + section_to_filename(section)
    json_path = os.path.join(output_dir, file_name)

    if not os.path.exists(output_dir):
        print(f"File not found for section: {section}")
        continue

    with open(json_path, "r", encoding="utf-8") as f:
        pages = json.load(f)

    filtered_pages = [p for p in pages if p["page"] not in exclude_pages]

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(filtered_pages, f, indent=2, ensure_ascii=False)

    print(f"Cleaned '{section}' — removed pages: {exclude_pages}")


Cleaned '3_The SDG Index and Dashboards' — removed pages: [35, 36, 37, 39, 40, 41, 46]
Cleaned '4_Government Efforts and Commitments for the SDGs' — removed pages: [60, 61, 63, 64, 67, 68, 70, 73, 75, 77, 78, 79, 80]


### Combine relevant text in a single file separated by section and page

- \n\n===== {section_title} =====\n\n
- page:

In [13]:
secton_json_path = os.path.join(path_data, "Processed Files (sections)")
output_dir_txt = os.path.join(path_data, "Processed Files (all text)")
os.makedirs(output_dir_txt, exist_ok=True)

extract_relevant_text_save_whole_text(secton_json_path, prefix, output_dir_txt)


Combined text file saved to: /content/TFM/Data/Processed Files (all text)/GLOBAL_SGD2023_full_relevant_text.txt


## GLOBAL SDG 2024 report

| Section                                                    | Pages  |
| ---------------------------------------------------------- | ------ |
| 0\_Unlocking the power of data for sustainable development | 4      |
| 1\_No poverty                                              | 2      |
| 2\_Zero hunger                                             | 2      |
| 3\_Good health and well-being                              | 4      |
| 4\_Quality education                                       | 2      |
| 5\_Gender equality                                         | 2      |
| 6\_Clean water and sanitation                              | 2      |
| 7\_Affordable and clean energy                             | 2      |
| 8\_Decent work and economic growth                         | 2      |
| 9\_Industry, innovation and infrastructure                 | 2      |
| 10\_Reduced inequalities                                   | 2      |
| 11\_Sustainable cities and communities                     | 2      |
| 12\_Responsible consumption and production                 | 2      |
| 13\_Climate action                                         | 2      |
| 14\_Life below water                                       | 2      |
| 15\_Life on land                                           | 2      |
| 16\_Peace, justice and strong institutions                 | 2      |
| 17\_Partnerships for the Goals                             | 2      |
| **Total**                                                  | **38** |


In [13]:
file_name = "GLOBAL_sustainable-development-report-2024.pdf"
prefix="GLOBAL_SGD2024_"

phrases_to_remove = [
    "The Sustainable Development Goals Report 2024\n"
]

### Define relevant sections to extract text from the Full report

In [14]:
relevant_sections_2024 = {
    "0_Unlocking the power of data for sustainable development": {"start_page": 6, "end_page": 9},
    "1_No poverty": {"start_page": 10, "end_page": 11},
    "2_Zero hunger": {"start_page": 12, "end_page": 13},
    "3_Good health and well-being": {"start_page": 14, "end_page": 17},
    "4_Quality education": {"start_page": 18, "end_page": 19},
    "5_Gender equality": {"start_page": 20, "end_page": 21},
    "6_Clean water and sanitation": {"start_page": 22, "end_page": 23},
    "7_Affordable and clean energy": {"start_page": 24, "end_page": 25},
    "8_Decent work and economic growth": {"start_page": 26, "end_page": 27},
    "9_Industry, innovation and infrastructure": {"start_page": 28, "end_page": 29},
    "10_Reduced inequalities": {"start_page": 30, "end_page": 31},
    "11_Sustainable cities and communities": {"start_page": 32, "end_page": 33},
    "12_Responsible consumption and production": {"start_page": 34, "end_page": 35},
    "13_Climate action": {"start_page": 36, "end_page": 37},
    "14_Life below water": {"start_page": 38, "end_page": 39},
    "15_Life on land": {"start_page": 40, "end_page": 41},
    "16_Peace, justice and strong institutions": {"start_page": 42, "end_page": 43},
    "17_Partnerships for the Goals": {"start_page": 44, "end_page": 45}
}

# Ensure the directory exists
os.makedirs(path_raw_data, exist_ok=True)

# Save dictionary to JSON file
json_path = os.path.join(path_raw_data, prefix + "sections_metadata.json")
with open(json_path, "w") as f:
    json.dump(relevant_sections_2024, f, indent=4)


### Extract relevant text by section in json with page and text

In [15]:
sections_metadata_file = os.path.join(path_raw_data, prefix + "sections_metadata.json")
pdf_path = os.path.join(path_raw_data, file_name)

# Output directory
output_dir = os.path.join(path_data, "Processed Files (sections)")
os.makedirs(output_dir, exist_ok=True)

# Generate .json files by section
extract_relevant_text_save_by_section(pdf_path, sections_metadata_file, output_dir,
                                      prefix=prefix,
                                      phrases_to_remove= phrases_to_remove)

Extraction complete. JSON files saved in: /content/TFM/Data/Processed Files (sections)


### Combine relevant text in a single file separated by section and page

- \n\n===== {section_title} =====\n\n
- page:

In [16]:
secton_json_path = os.path.join(path_data, "Processed Files (sections)")
output_dir_txt = os.path.join(path_data, "Processed Files (all text)")
os.makedirs(output_dir_txt, exist_ok=True)

extract_relevant_text_save_whole_text(secton_json_path, prefix, output_dir_txt)

Combined text file saved to: /content/TFM/Data/Processed Files (all text)/GLOBAL_SGD2024_full_relevant_text.txt


## GLOBAL SDG 2025 report

| Section                                                | Pages  |
| ------------------------------------------------------ | ------ |
| 1\_Executive Summary                                   | 2      |
| 2\_Financing for Development                           | 9      |
| 3\_The SDG Index and Dashboards                        | 6      |
| 4\_Commitment to the SDGs and UN-Based Multilateralism | 10     |
| **Total**                                              | **27** |


In [17]:
file_name = "GLOBAL_sustainable-development-report-2025.pdf"
prefix="GLOBAL_SGD2025_"

phrases_to_remove=['\nSustainable Development Report 2025',
                   '\n   Financing Sustainable Development to 2030 and Mid-Century\n']

### Define relevant sections to extract text from the Full report

In [18]:
relevant_sections_2025 = {
    "1_Executive Summary": {"start_page": 10, "end_page": 11},
    "2_Financing for Development": {"start_page": 13, "end_page": 21},
    "3_The SDG Index and Dashboards": {"start_page": 23, "end_page": 33},
    "4_Commitment to the SDGs and UN-Based Multilateralism": {"start_page": 45, "end_page": 59}
}


# Ensure the directory exists
os.makedirs(path_raw_data, exist_ok=True)

# Save dictionary to JSON file
json_path = os.path.join(path_raw_data, prefix + "sections_metadata.json")
with open(json_path, "w") as f:
    json.dump(relevant_sections_2025, f, indent=4)


### Extract relevant text by section in json with page and text

In [19]:
sections_metadata_file = os.path.join(path_raw_data, prefix + "sections_metadata.json")
pdf_path = os.path.join(path_raw_data, file_name)

# Output directory
output_dir = os.path.join(path_data, "Processed Files (sections)")
os.makedirs(output_dir, exist_ok=True)

# Generate .json files by section
extract_relevant_text_save_by_section(pdf_path, sections_metadata_file, output_dir,
                                      prefix=prefix,
                                      phrases_to_remove=phrases_to_remove)

Extraction complete. JSON files saved in: /content/TFM/Data/Processed Files (sections)


In [20]:
### Manual corrections (exclude pages with only tables or infographics to avoid wrong interpretations)

# Pages to exclude per section
pages_to_exclude = {
    "3_The SDG Index and Dashboards": [24, 25, 27, 28, 29],
    "4_Commitment to the SDGs and UN-Based Multilateralism": [47, 48, 49, 52, 58],
}

# Helper to sanitize section name into filename
def section_to_filename(section_title):
    return section_title.replace(" ", "_").replace(":", "").replace("/", "-") + ".json"

# Process each targeted section
for section, exclude_pages in pages_to_exclude.items():
    file_name = prefix + section_to_filename(section)
    json_path = os.path.join(output_dir, file_name)

    if not os.path.exists(output_dir):
        print(f"File not found for section: {section}")
        continue

    with open(json_path, "r", encoding="utf-8") as f:
        pages = json.load(f)

    filtered_pages = [p for p in pages if p["page"] not in exclude_pages]

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(filtered_pages, f, indent=2, ensure_ascii=False)

    print(f"Cleaned '{section}' — removed pages: {exclude_pages}")


Cleaned '3_The SDG Index and Dashboards' — removed pages: [24, 25, 27, 28, 29]
Cleaned '4_Commitment to the SDGs and UN-Based Multilateralism' — removed pages: [47, 48, 49, 52, 58]


### Combine relevant text in a single file separated by section and page

- \n\n===== {section_title} =====\n\n
- page:

In [21]:
secton_json_path = os.path.join(path_data, "Processed Files (sections)")
output_dir_txt = os.path.join(path_data, "Processed Files (all text)")
os.makedirs(output_dir_txt, exist_ok=True)

extract_relevant_text_save_whole_text(secton_json_path, prefix, output_dir_txt)

Combined text file saved to: /content/TFM/Data/Processed Files (all text)/GLOBAL_SGD2025_full_relevant_text.txt


## OCDE paper: Localising the SDGs in a changing 2024

| Section                                                                        | Pages  |
| ------------------------------------------------------------------------------ | ------ |
| 1\_Preface                                                                     | 2      |
| 2\_Executive Summary                                                           | 3      |
| 3\_Introduction                                                                | 3      |
| 4\_Snapshot of SDG Implementation in Cities and Regions                        | 6      |
| 5\_The Role of the SDGs for COVID-19 Recovery in Cities and Regions            | 3      |
| 6\_Impact of Current Crises on Key Policy Areas and SDGs                       | 14     |
| 7\_Annex A: Examples of Policies and Actions Implemented by Cities and Regions | 2      |
| **Total**                                                                      | **33** |


In [22]:
file_name = "OCDE_Localising the SDGs in a changing_2024.pdf"
prefix="OCDE_2024_"

phrases_to_remove = ['\nLOCALISING THE SDGS IN A CHANGING LANDSCAPE © OECD/SDSN 2024 \n']

### Define relevant sections to extract text from the Full report

In [23]:
relevant_sections_OCDE_2024 = {
    "1_Preface": {"start_page": 4, "end_page": 5},
    "2_Executive Summary": {"start_page": 10, "end_page": 12},
    "3_Introduction": {"start_page": 13, "end_page": 15},
    "4_Snapshot of SDG Implementation in Cities and Regions": {"start_page": 17, "end_page": 22},
    "5_The Role of the SDGs for COVID-19 Recovery in Cities and Regions": {"start_page": 24, "end_page": 26},
    "6_Impact of Current Crises on Key Policy Areas and SDGs": {"start_page": 27, "end_page": 40},
    "7_Annex A: Examples of Policies and Actions Implemented by Cities and Regions": {"start_page": 42, "end_page": 43}
}

# Ensure the directory exists
os.makedirs(path_raw_data, exist_ok=True)

# Save dictionary to JSON file
json_path = os.path.join(path_raw_data, prefix + "sections_metadata.json")
with open(json_path, "w") as f:
    json.dump(relevant_sections_OCDE_2024, f, indent=4)


### Extract relevant text by section in json with page and text

In [24]:
sections_metadata_file = os.path.join(path_raw_data, prefix + "sections_metadata.json")
pdf_path = os.path.join(path_raw_data, file_name)

# Output directory
output_dir = os.path.join(path_data, "Processed Files (sections)")
os.makedirs(output_dir, exist_ok=True)

# Generate .json files by section
extract_relevant_text_save_by_section(pdf_path, sections_metadata_file, output_dir,
                                      prefix=prefix,
                                      phrases_to_remove= phrases_to_remove)

Extraction complete. JSON files saved in: /content/TFM/Data/Processed Files (sections)


### Combine relevant text in a single file separated by section and page

- \n\n===== {section_title} =====\n\n
- page:

In [25]:
secton_json_path = os.path.join(path_data, "Processed Files (sections)")
output_dir_txt = os.path.join(path_data, "Processed Files (all text)")
os.makedirs(output_dir_txt, exist_ok=True)

extract_relevant_text_save_whole_text(secton_json_path, prefix, output_dir_txt)

Combined text file saved to: /content/TFM/Data/Processed Files (all text)/OCDE_2024_full_relevant_text.txt


## G20 paper: Addressing negative spillover effects for SDG

| Section                                                                              | Pages  |
| ------------------------------------------------------------------------------------ | ------ |
| 1\_Abstract                                                                          | 1      |
| 2\_Diagnosis of the Issue: Addressing Negative Spillover Effects for SDG Achievement | 4      |
| 3\_Recommendations                                                                   | 4      |
| 4\_Scenario of Outcome: In Support of a Successful Summit of the Future              | 1      |
| **Total**                                                                            | **10** |

---


In [26]:
file_name = "G20_Addressing negative spillover effects for SDG.pdf"
prefix="G20_2024_"

### Define relevant sections to extract text from the Full report

In [27]:
relevant_sections_G20_2024 = {
    "1_Abstract": {"start_page": 2, "end_page": 2},
    "2_Diagnosis of the Issue: Addressing Negative Spillover Effects for SDG Achievement": {"start_page": 3, "end_page": 6},
    "3_Recommendations": {"start_page": 7, "end_page": 10},
    "4_Scenario of Outcome: In Support of a Successful Summit of the Future": {"start_page": 11, "end_page": 11}
}

# Ensure the directory exists
os.makedirs(path_raw_data, exist_ok=True)

# Save dictionary to JSON file
json_path = os.path.join(path_raw_data, prefix + "sections_metadata.json")
with open(json_path, "w") as f:
    json.dump(relevant_sections_G20_2024, f, indent=4)


### Extract relevant text by section in json with page and text

In [28]:
sections_metadata_file = os.path.join(path_raw_data, prefix + "sections_metadata.json")
pdf_path = os.path.join(path_raw_data, file_name)

# Output directory
output_dir = os.path.join(path_data, "Processed Files (sections)")
os.makedirs(output_dir, exist_ok=True)

# Generate .json files by section
extract_relevant_text_save_by_section(pdf_path, sections_metadata_file, output_dir, prefix=prefix)

Extraction complete. JSON files saved in: /content/TFM/Data/Processed Files (sections)


### Combine relevant text in a single file separated by section and page

- \n\n===== {section_title} =====\n\n
- page:

In [29]:
secton_json_path = os.path.join(path_data, "Processed Files (sections)")
output_dir_txt = os.path.join(path_data, "Processed Files (all text)")
os.makedirs(output_dir_txt, exist_ok=True)

extract_relevant_text_save_whole_text(secton_json_path, prefix, output_dir_txt)

Combined text file saved to: /content/TFM/Data/Processed Files (all text)/G20_2024_full_relevant_text.txt


In [14]:
!git add .
!git commit -m "Fix relevant pages in 2023 report"
!git push origin main  # or 'master' or your branch name

hint: You've added another git repository inside your current repository.
hint: Clones of the outer repository will not contain the contents of
hint: the embedded repository and will not know how to obtain it.
hint: If you meant to add a submodule, use:
hint: 
hint: 	git submodule add <url> TFM
hint: 
hint: If you added this path by mistake, you can remove it from the
hint: index with:
hint: 
hint: 	git rm --cached TFM
hint: 
hint: See "git help submodule" for more information.
[main 69155b3] Fix relevant pages in 2023 report
Enumerating objects: 21, done.
Counting objects: 100% (16/16), done.
Delta compression using up to 2 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 873 bytes | 873.00 KiB/s, done.
Total 9 (delta 6), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (6/6), completed with 5 local objects.
To https://github.com/camipalo/TFM.git
   9e82c35..69155b3  main -> main
